# 21 — Fannie Mae retraining feasibility and reliability
## Stage 1 — Training population and maturity audit
Check which earlier Fannie loans could enter a January 2023 update under reporting and full-window maturity proxies. Use the same 2023 evaluation loans for every model. Stage 1 is reconstructed and verified against the reviewed audit before Stage 2 fitting.

Primary planned experiment: a Fannie-only 2015–2017 XGBoost baseline versus Fannie-only expanding updates, using the approved 22 predictors, the Notebook 16 XGBoost settings fixed in advance and preprocessing fitted on Fannie development loans only and frozen for both updates. This separates retraining from a change in training agency. The existing Freddie-trained frozen companion from Notebook 19 will be an external reference, not the within-agency frozen baseline.

Policy A is a conservative whole-cohort **full-window** version of Notebook 13's termination-aware policy. Notebook 18D does not retain each loan's terminal completion month; do not infer it from the first snapshot. Policy B follows the original individual full-window rule. Both policies retain the complete development sample only if it satisfies the readiness proxy. Sensitivity lags of 3 and 6 months are audited, not optimized. No FICO predictor, weighting, or tuning is added.

## 1. Verify the reviewed data and model specification
Locate the exact reviewed manifests rather than selecting the latest run.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, zipfile, sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
FANNIE = ROOT / 'data/processed/Fannie_Mae'
COMPANION = ROOT / 'data/processed/Freddie_Mac/companion_models_2015_2017/20261125T142258634663Z'
DECISION = pd.Period('2023-01', freq='M')
LAGS = [0, 3, 6]
TARGET = 'TARGET_PRIMARY_COMPOSITE'
KEY = 'primary_22__xgboost'

def require(ok, message):
    if not ok:
        raise ValueError(message)

def sha(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def reviewed(folder, pattern, digest):
    matches = sorted(p for p in folder.glob(pattern) if sha(p) == digest)
    require(bool(matches), f'Reviewed manifest not found in {folder}')
    return matches[0]

p18 = reviewed(FANNIE / 'notebook_18d_consolidated_v1/final', '*/manifest.json', '41125a32393ce4170dbd9ee75399da6b616975298cbb92dd145a6ebad5f24fbc')
p19 = reviewed(FANNIE / 'notebook_19_multi_year_model_reliability/final', '*/model_evaluation_manifest.json', '5786e3b73578a0abf47766c1fba753984b054a29265fea5d430971c7d9cb26ac')
p16 = COMPANION / 'model_manifest.json'
require(sha(p16) == '0831ffa3f152a46c0d6745cec758fd3dc98bf6b416dc57b3d9ccb8461afbfac7', 'Notebook 16 manifest changed.')
m18, m19, m16 = [json.loads(p.read_text(encoding='utf-8')) for p in [p18,p19,p16]]
DATA = p18.parent / 'fannie_2015_2023_sampled_dataset.parquet'
PRED = p19.parent / 'fannie_2018_2023_predictions.parquet'
require(sha(DATA) == m18['dataset_sha256'] == m19['dataset_sha256'], 'Dataset hash differs.')
require(sha(PRED) == m19['output_sha256'][PRED.name], 'Notebook 19 predictions changed.')
require(m19['companion_manifest_sha256'] == sha(p16), 'Companion identity differs.')
features = m16['specifications']['primary_22']
require(len(features) == 22 and features == m19['predictors'] and set(features) == set(m18['primary_features']), 'Predictors differ.')
record = m16['models'][KEY]
require(record['selected_params'] == {'learning_rate':0.05,'max_depth':2,'n_estimators':200}, 'Selected XGBoost settings differ.')
protected = {p:sha(p) for p in [p18,p19,p16,DATA,PRED]}
for name, digest in m16['source_sha256'].items():
    path = ROOT / name
    require(sha(path) == digest, f'Source changed: {name}')
    protected[path] = digest
for name, digest in [(record['artifact'],record['sha256']), (record['feature_contract'],record['feature_contract_sha256'])]:
    path = COMPANION / name
    require(sha(path) == digest, f'Companion artifact changed: {name}')
    protected[path] = digest
print('Reviewed inputs verified. No model loaded or fitted.')

## 2. Reconcile loan populations
Keep the existing primary eligibility and composite target. Check exact evaluation keys and outcomes against Notebook 19.

In [ ]:
cols = list(dict.fromkeys(['loan_id','origination_year','ORIGINATION_COHORT','half_year','first_payment','report','acquisition_file','primary','strict',TARGET,'MI_MAPPING_INCONSISTENT']+features))
raw = pd.read_parquet(DATA, columns=cols)
require(len(raw)==450000 and raw.loan_id.notna().all() and raw.loan_id.is_unique, 'Sample keys differ.')
require(raw.groupby('origination_year').size().to_dict()==dict.fromkeys(range(2015,2024),50000), 'Annual sample differs.')
require(pd.api.types.is_bool_dtype(raw.primary) and raw.primary.notna().all(), 'Invalid eligibility.')
e = raw.loc[raw.primary].copy().reset_index(drop=True)
require(len(e)==430569 and e[TARGET].isin([0,1]).all() and int(e[TARGET].sum())==7223, 'Eligible population differs.')
require(not e.MI_MAPPING_INCONSISTENT.any(), 'MI mapping issue requires review.')
periods = [f'{y}-H{h}' for y in range(2015,2024) for h in (1,2)]
require(set(e.half_year)==set(periods), 'Half-year coverage differs.')
dev = e.origination_year.le(2017)
test = e.origination_year.eq(2023)
require(int(dev.sum())==142506 and int(e.loc[dev,TARGET].sum())==944, 'Development counts differ.')
require(int(test.sum())==47811 and int(e.loc[test,TARGET].sum())==592, 'Evaluation counts differ.')
old = pd.read_parquet(PRED, columns=['loan_id','half_year','y_true',KEY])
require(old.loan_id.is_unique and len(old)==288063, 'Saved prediction keys differ.')
a = e.loc[e.origination_year.ge(2018),['loan_id','half_year',TARGET]].rename(columns={TARGET:'y_true'}).sort_values('loan_id').reset_index(drop=True)
b = old[['loan_id','half_year','y_true']].sort_values('loan_id').reset_index(drop=True)
pd.testing.assert_frame_equal(a,b,check_dtype=False)
require(np.isfinite(old[KEY]).all() and old[KEY].between(0,1).all(), 'Invalid reference probabilities.')
print('Development:', int(dev.sum()), 'Evaluation:', int(test.sum()), 'events:', int(e.loc[test,TARGET].sum()))

## 3. Reconstruct maturity proxies
A label is treated as ready at first payment + 24 months, and the first report must precede the decision. For lag sensitivity, add the same lag to both reporting and maturity readiness. The acquisition quarter is audited separately, not treated as publication evidence. No terminal event is used to accelerate readiness.

In [ ]:
def parse_month(s):
    return pd.to_datetime(s.astype('string'),format='%m%Y',errors='coerce').dt.to_period('M')
fp = parse_month(e.first_payment)
report = parse_month(e.report)
require(fp.notna().all() and report.notna().all(), 'Eligible loan dates missing or invalid.')
require(e.acquisition_file.astype('string').str.fullmatch(r'20\d{2}Q[1-4]').all(), 'Acquisition quarter invalid.')
acq_year = e.acquisition_file.str[:4].astype(int)
acq_quarter = e.acquisition_file.str[-1].astype(int)
acq_end = pd.to_datetime(dict(year=acq_year,month=acq_quarter*3,day=1)).dt.to_period('M')
full_ready = fp + 24
print('Latest development full-window readiness:', full_ready[dev].max())
print('Latest development first report:', report[dev].max())
print('Decision month:', DECISION)
# A/B use only cohort, payment and report dates after eligibility is fixed.
# Acquisition-period end is a separate descriptive conservative bound.
prior = e.origination_year.between(2018,2022)
cohort_rows, policy_rows, check_rows, recency_rows, acquisition_rows = [], [], [], [], []
fingerprints = {}
for lag in LAGS:
    ready = (full_ready + lag <= DECISION) & (report + 1 + lag <= DECISION)
    whole = ready.groupby(e.half_year).transform('all')
    masks = {'A_whole_cohort_full_window':dev | (prior & whole),
             'B_full_window_individual':dev | (prior & ready)}
    for period, g in e.groupby('half_year',sort=True):
        idx = g.index
        cohort_rows.append(dict(lag_months=lag,period=period,eligible=len(g),events=int(g[TARGET].sum()),
            full_window_ready=int(ready.loc[idx].sum()),whole_cohort_ready=bool(whole.loc[idx].all()),
            latest_full_window_ready=str((full_ready.loc[idx]+lag).max()),latest_first_report=str(report.loc[idx].max()),
            acquisition_quarter_not_ended_by_decision=int((acq_end.loc[idx]>=DECISION).sum())))
    for policy, mask in masks.items():
        add=mask & ~dev
        selected=e.loc[mask]
        key_text='\n'.join(sorted(selected.loan_id.astype(str)))
        fingerprints[f'{policy}|lag={lag}']=hashlib.sha256(key_text.encode()).hexdigest()
        for sample,sm in [('development',dev),('additional_history',add),('update',mask),('evaluation',test)]:
            g=e.loc[sm]
            policy_rows.append(dict(lag_months=lag,policy=policy,sample=sample,loans=len(g),events=int(g[TARGET].sum()),
                prevalence=float(g[TARGET].mean()) if len(g) else np.nan,periods=';'.join(sorted(g.half_year.unique()))))
        for period,g in e.groupby('half_year',sort=True):
            s=g.loc[mask.loc[g.index]]
            recency_rows.append(dict(lag_months=lag,policy=policy,period=period,eligible=len(g),selected=len(s),selected_events=int(s[TARGET].sum())))
        acquisition_rows.append(dict(lag_months=lag,policy=policy,selected_loans=len(selected),
             selected_acquisition_quarter_not_ended_by_decision=int((acq_end[mask]>=DECISION).sum()),
             selected_full_window_not_ready=int((~ready[mask]).sum())))
        checks={'development retained':bool(mask[dev].all()),'development ready':bool(ready[dev].all()),
                'selected ready':bool(ready[mask].all()),'no 2023 loans in training':not bool((mask&test).any()),
                'additional history present':bool(add.any()),'both training classes':selected[TARGET].nunique()==2,
                'both evaluation classes in each half-year':bool(e.loc[test].groupby('half_year')[TARGET].nunique().eq(2).all()),
                'acquisition quarter ended before decision':bool((acq_end[mask]<DECISION).all())}
        for check,passed in checks.items():
            check_rows.append(dict(lag_months=lag,policy=policy,check=check,passed=bool(passed)))
    require((~masks['A_whole_cohort_full_window'] | masks['B_full_window_individual']).all(), 'Whole-cohort rule should be nested here.')
cohort_audit=pd.DataFrame(cohort_rows)
policy_summary=pd.DataFrame(policy_rows)
policy_cohorts=pd.DataFrame(recency_rows)
checks=pd.DataFrame(check_rows)
acquisition_audit=pd.DataFrame(acquisition_rows)
primary_pass=bool(checks.loc[checks.lag_months.eq(0),'passed'].all())
display(policy_summary.query('lag_months == 0'))
display(checks)
print('Primary audit:', 'passed under proxies' if primary_pass else 'blocked — review required')

## 4. Verify the reviewed Stage 1 audit
Reconstruct the exact zero-lag training selections and verify the loan-key fingerprints. The 3- and 6-month scenarios remain feasibility sensitivities only.

In [ ]:
audit_path = reviewed(FANNIE/'notebook_21_external_retraining'/'stage1', '*/manifest.json', 'bc53e3e89f4eabf30e8b99625bc4fd5167092375bbbf3fb48d824a8a400bf3d6')
audit = json.loads(audit_path.read_text(encoding='utf-8'))
require(audit['status']=='structurally_feasible_under_proxies_pending_review' and checks.passed.all(), 'Feasibility audit failed.')
require(audit['selected_key_sha256']==fingerprints, 'Training loan fingerprints differ.')
require(audit['input_sha256']=={p.relative_to(ROOT).as_posix():h for p,h in protected.items()}, 'Audit input hashes differ.')
protected[audit_path]=sha(audit_path)
for name,h in audit['output_sha256'].items():
    p=audit_path.parent/name
    require(sha(p)==h, f'Audit output changed: {name}')
    protected[p]=h
ready=(full_ready<=DECISION)&(report+1<=DECISION)
whole=ready.groupby(e.half_year).transform('all')
train_masks={'frozen':dev,'policy_a':dev|(prior&whole),'policy_b':dev|(prior&ready)}
expected={'frozen':(142506,944),'policy_a':(237246,4821),'policy_b':(281219,5391)}
for name,mask in train_masks.items():
    require((int(mask.sum()),int(e.loc[mask,TARGET].sum()))==expected[name], f'{name}: training totals differ')
    require(not (mask&test).any() and ready[mask].all(), f'{name}: maturity or overlap check failed')
print('Exact reviewed training selections verified.')

## 5. Fit the baseline and two updates
Fit the feature contract and encoder only on Fannie 2015–2017. Reuse the fitted preprocessing unchanged for both updates. The XGBoost settings are inherited from the Freddie companion model, not selected using Fannie evaluation results.

In [ ]:
import joblib, sklearn, xgboost
from copy import deepcopy
from sklearn.base import clone
from sklearn.pipeline import Pipeline
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.freddie_companion import fit_contract, apply_contract, model_pipeline, development_bin_edges, calibration_table
from src.fannie_external_evaluation import METRICS, summarize, bootstrap_group, intervals, stable_seed
for name,version in [('pandas',pd.__version__),('numpy',np.__version__),('scikit_learn',sklearn.__version__),('xgboost',xgboost.__version__),('joblib',joblib.__version__)]:
    require(m16[name]==version,f'Use original environment: {name} differs.')
source=e.assign(**{'SOURCE YEAR':e.origination_year})
contract=fit_contract(source,'primary_22',[2015,2016,2017])
require(contract['predictors']==features and contract['training_rows']==142506,'Development contract differs.')
X=apply_contract(source,contract)
y=e[TARGET].astype(int)
models={}
baseline=model_pipeline('xgboost',record['selected_params'],contract)
baseline.fit(X.loc[dev],y.loc[dev])
models['frozen']=baseline
prep=baseline.named_steps['prepare']
for name in ['policy_a','policy_b']:
    mask=train_masks[name]
    estimator=clone(baseline.named_steps['model'])
    estimator.fit(prep.transform(X.loc[mask]),y.loc[mask])
    models[name]=Pipeline([('prepare',deepcopy(prep)),('model',estimator)])
    print('Fit complete:',name,flush=True)
print('All three models fitted; preprocessing learned only from development.')

## 6. Score the common 2023 population
Use the same loans for all three models. The Freddie-trained companion is reported separately as context. Risk gap means predicted minus observed. Positive improvement favors the updated model.

In [ ]:
evaluation=e.loc[test,['loan_id','half_year',TARGET]].rename(columns={TARGET:'y_true'}).reset_index(drop=True)
for name,model in models.items():
    evaluation[name]=model.predict_proba(X.loc[test])[:,1]
    require(np.isfinite(evaluation[name]).all() and evaluation[name].between(0,1).all(),f'Invalid predictions: {name}')
evaluation=evaluation.merge(old[['loan_id',KEY]].rename(columns={KEY:'freddie_reference'}),on='loan_id',validate='one_to_one')
require(len(evaluation)==47811 and evaluation.y_true.sum()==592,'Evaluation totals changed.')
edges=development_bin_edges(baseline.predict_proba(X.loc[dev])[:,1])
model_names=list(models)
frozen_edges={name:edges for name in model_names+['freddie_reference']}
metric_parts=[]; calibration_parts=[]; context_parts=[]; differences=[]; draw_parts=[]; interval_parts=[]
metrics=['roc_auc','average_precision','brier','absolute_risk_gap']
for period,frame in [('2023-H1',evaluation.query("half_year=='2023-H1'")),('2023-H2',evaluation.query("half_year=='2023-H2'")),('2023',evaluation)]:
    yy=frame.y_true.to_numpy(dtype=int)
    pred={name:frame[name].to_numpy(dtype=float) for name in model_names}
    point=summarize(yy,pred,frozen_edges).assign(period=period)
    metric_parts.append(point)
    context_parts.append(summarize(yy,{'freddie_reference':frame.freddie_reference.to_numpy()},frozen_edges).assign(period=period))
    for name in model_names:
        calibration_parts.append(calibration_table(yy,pred[name],edges).assign(period=period,model=name))
    boot=bootstrap_group(yy,pred,frozen_edges,draws=1000,seed=stable_seed('Notebook21:'+period,42),progress=lambda done,total: print(period,done,'/',total,flush=True))
    pv=point.set_index('model').loc[model_names,METRICS].to_numpy()
    for j,name in enumerate(model_names):
        draw_parts.append(pd.DataFrame(boot['draws'][:,j],columns=METRICS).assign(period=period,model=name,draw_index=np.arange(1000)))
        interval_parts.append(intervals(boot['draws'][:,j],pv[j],dict(period=period,model=name)))
    for left,right in [('policy_a','frozen'),('policy_b','frozen'),('policy_b','policy_a')]:
        i,j=model_names.index(left),model_names.index(right)
        for metric in metrics:
            k=METRICS.index(metric); sign=1 if metric in ['roc_auc','average_precision'] else -1
            draws=sign*(boot['draws'][:,i,k]-boot['draws'][:,j,k])
            require(np.isfinite(draws).all(),'Invalid paired draw')
            lo,hi=np.quantile(draws,[.025,.975])
            differences.append(dict(period=period,comparison=left+'_minus_'+right,metric=metric,improvement=sign*(pv[i,k]-pv[j,k]),ci_low=lo,ci_high=hi,valid_draws=len(draws)))
performance=pd.concat(metric_parts,ignore_index=True)
paired=pd.DataFrame(differences)
calibration=pd.concat(calibration_parts,ignore_index=True)
context=pd.concat(context_parts,ignore_index=True).drop(columns=['ece_development_bins']) # Fannie bins are not the Freddie model's own reference
require(len(performance)==9 and len(paired)==36,'Evaluation coverage differs.')
for row in performance.itertuples():
    bins=calibration.query('period==@row.period and model==@row.model')
    require(bins.n.sum()==row.n and bins.events.sum()==row.events,'Calibration totals differ.')
display(performance.round(6)); display(paired.round(6))

## 7. Compare calibration and save results
The paired intervals condition on the trained models and do not capture training uncertainty or shared economic shocks. No policy selection or prospective claim is made. The 3/6-month lag scenarios are audited for feasibility only, not fitted.

In [ ]:
import matplotlib.pyplot as plt
OUT=FANNIE/'notebook_21_external_retraining'/'final'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
require(all(sha(p)==h for p,h in protected.items()),'Protected input changed.')
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'figures').mkdir()
for name,model in models.items():
    joblib.dump(model,OUT/f'{name}_xgboost.joblib')
    loaded=joblib.load(OUT/f'{name}_xgboost.joblib')
    np.testing.assert_allclose(loaded.predict_proba(X.loc[test].head(50)),model.predict_proba(X.loc[test].head(50)),rtol=0,atol=0)
(OUT/'fannie_development_contract.json').write_text(json.dumps(contract,indent=2,allow_nan=False),encoding='utf-8')
(OUT/'frozen_calibration_bins.json').write_text(json.dumps(edges),encoding='utf-8')
evaluation.to_parquet(OUT/'local_2023_predictions.parquet',index=False)
common=evaluation.copy()
common['bin']=pd.qcut(common.frozen,10,labels=False,duplicates='drop')
common_bins=common.groupby('bin').agg(loans=('y_true','size'),events=('y_true','sum'),observed=('y_true','mean'),frozen=('frozen','mean'),policy_a=('policy_a','mean'),policy_b=('policy_b','mean')).reset_index()
require(common_bins.loans.sum()==47811 and common_bins.events.sum()==592,'Common score bins differ.')
fig,ax=plt.subplots(figsize=(9,5))
for name in ['observed','frozen','policy_a','policy_b']:
    ax.plot(common_bins.bin+1,100*common_bins[name],marker='o',label=name)
ax.set(xlabel='Frozen model score group (low to high)',ylabel='Risk (%)',title='2023 Fannie Mae: observed and predicted risk on identical loan groups');ax.legend();fig.tight_layout();fig.savefig(OUT/'figures/common_calibration.png',dpi=180);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(8,5))
p=performance.query("period=='2023'").set_index('model').loc[model_names]
ax.bar(np.arange(3),100*p.mean_predicted);ax.axhline(100*p.prevalence.iloc[0],linestyle='--',label='Observed rate');ax.set(xticks=np.arange(3),xticklabels=model_names,ylabel='Risk (%)',title='2023 Fannie Mae — mean predicted risk');ax.legend();fig.tight_layout();fig.savefig(OUT/'figures/mean_risk.png',dpi=180);plt.show();plt.close(fig)
tables={'training_policy_summary.csv':policy_summary,'policy_cohort_selection.csv':policy_cohorts,'feasibility_checks.csv':checks,'cohort_maturity_audit.csv':cohort_audit,'performance.csv':performance,'paired_improvement_intervals.csv':paired,'metric_intervals.csv':pd.concat(interval_parts,ignore_index=True),'bootstrap_draws.csv':pd.concat(draw_parts,ignore_index=True),'calibration_frozen_bins.csv':calibration,'common_score_groups.csv':common_bins,'freddie_reference_context.csv':context}
for name,frame in tables.items(): frame.to_csv(OUT/name,index=False)
require(all(sha(p)==h for p,h in protected.items()),'Protected input changed during fitting.')
manifest=dict(stage='21_final',status='evaluation_complete_pending_review',decision_month=str(DECISION),primary_lag=0,lag_sensitivity='feasibility only',training_counts=expected,selected_key_sha256=fingerprints,predictors=features,target=TARGET,horizon_months=24,params=record['selected_params'],bootstrap_draws=1000,risk_gap='predicted minus observed',paired_difference='positive improvement favors first model',fannie_baseline=True,preprocessing_frozen=True,policy_a='whole cohort full window; approximation to Freddie termination-aware rule',historical_publication_verified=False,prospective_validation=False,input_sha256={p.relative_to(ROOT).as_posix():h for p,h in protected.items()},output_sha256={p.relative_to(OUT).as_posix():sha(p) for p in OUT.rglob('*') if p.is_file()})
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2,allow_nan=False),encoding='utf-8')
review=OUT/'notebook_21_final_review.zip'
with zipfile.ZipFile(review,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.suffix in {'.csv','.json','.png'}: z.write(p,p.relative_to(OUT).as_posix())
with zipfile.ZipFile(review) as z: require(z.testzip() is None,'ZIP integrity failed.')
print('UPLOAD ONLY:',review)